# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates the de-id module applying Format-Preserving Encryption (FPE) for SSN/MRN, deterministic date-shifting (+/- 30 days), and free-text Safe Harbor 18 redaction.

In [ ]:
import os
import hashlib
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, expr
from pyspark.sql.types import StringType

os.environ['SPARK_VERSION'] = '3.5'
os.environ['EHDIP_HASH_SALT'] = 'test_salt_123'

spark = SparkSession.builder \
    .appName("Day04_PHI_DeID") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Synthetic Patient Data
data = [
    ("P001", "123-45-6789", "MRN991", "1980-05-15", "Patient called regarding test results on 2023-01-01. SSN is 123-45-6789."),
    ("P002", "987-65-4321", "MRN992", "1992-11-20", "No issues reported.")
]

columns = ["patient_id", "ssn", "mrn", "birth_date", "notes"]
df_raw = spark.createDataFrame(data, columns)

print("Raw Data:")
df_raw.show(truncate=False)


In [ ]:
def get_hash_salt():
    return os.environ.get("EHDIP_HASH_SALT", "default")

def _fpe_mask(value):
    if not value:
        return value
    salt = get_hash_salt()
    salted_value = f"{value}{salt}".encode('utf-8')
    return hashlib.sha256(salted_value).hexdigest()

fpe_udf = udf(_fpe_mask, StringType())

# Apply De-Identification
df_deid = df_raw \
    .withColumn("ssn_masked", fpe_udf(col("ssn"))) \
    .withColumn("mrn_masked", fpe_udf(col("mrn"))) \
    .withColumn("date_shift_days", expr("pmod(conv(substr(md5(patient_id), 1, 15), 16, 10), 61) - 30")) \
    .withColumn("birth_date_shifted", expr("date_add(cast(birth_date as date), cast(date_shift_days as int))")) \
    .withColumn("notes_redacted", expr("regexp_replace(notes, '\\\\b\\\\d{3}-\\\\d{2}-\\\\d{4}\\\\b', '[REDACTED_SSN]')")) \
    .select("patient_id", "ssn_masked", "mrn_masked", "birth_date_shifted", "notes_redacted")

print("De-Identified Data:")
df_deid.show(truncate=False)
